# Graph-Embedding-Based Structured CNN Pruning

Runs the full pipeline from `src/pipeline.py`: train a baseline model, extract per-filter activation statistics, build a filter-similarity graph, learn graph embeddings, cluster filters by redundancy, prune (our method + an L2-norm magnitude baseline), recalibrate each pruned model, and compare all three.

This notebook is a thin wrapper around `run_pipeline(config)` — all the actual logic lives in the repo (`src/`), verified there. Every setting below is a config field; see the repo's `README.md` "Config reference" section for what each one means and why it's tunable rather than hardcoded.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/TDShemTov/pruningconvnetspaper.git
%cd pruningconvnetspaper
!pip install -q -r requirements.txt

## 2. Configure the run

Swap points worth trying: `model_name` (`simplecnn` / `resnet18` / `resnet34` / `resnet50` / `vgg11_bn`...`vgg19_bn` / `densenet121`/`169`/`201`), `dataset_name` (any of the 20 in `src/data/datasets.py`'s `DATASET_REGISTRY`), `graph_embedding_method` (`node2vec` / `spectral` / `raw` / `diffusion` / `gcn` — see README for what each one uses and its own config class), and `graph_config`'s `same_layer_only`/`cross_layer_threshold` (same-layer vs. global vs. hybrid topology — see CLAUDE.md Step 4).

In [ ]:
import torch

from src.pipeline import PipelineConfig, train_base, prune_and_compare
from src.data.datasets import SplitConfig
from src.train import TrainConfig
from src.embedding import ActivationConfig
from src.graph import (
    GraphConfig,
    Node2VecConfig,
    SpectralEmbedConfig,
    RawEmbedConfig,
    DiffusionEmbedConfig,
    GCNEmbedConfig,
)
from src.clustering import ClusterConfig
from src.eval import TimingConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# --- reproducibility: True = today's fixed-seed behavior (same data split /
# weight init / shuffling every run). False = no manual_seed call at all --
# PyTorch/numpy draw from their own OS-entropy default RNG, so back-to-back
# runs are genuinely independent (useful for a poor-man's seed-robustness
# check: rerun this notebook a few times with fixed_seed=False and see how
# much ours_vs_baseline/l2_vs_baseline actually moves).
fixed_split = True
fixed_seed = True
split_seed = 42 if fixed_split else None
train_seed = 42 if fixed_seed else None

# --- graph embedding: pick the method, then tune its hyperparams below. Only
# the config matching `graph_embedding_method` is actually used (see README's
# "Config reference" table) -- all five are defined here together so
# switching methods is just editing the matching block, not hunting for it.
graph_embedding_method = "node2vec"  # "node2vec" | "spectral" | "raw" | "diffusion" | "gcn"

node2vec_config = Node2VecConfig(embed_dim=64, walk_length=30, num_walks=50, epochs=5, device=device)
spectral_config = SpectralEmbedConfig(embed_dim=64)
raw_embed_config = RawEmbedConfig(embed_dim=64)
diffusion_config = DiffusionEmbedConfig(embed_dim=64, k=2, use_edge_weights=True)
gcn_config = GCNEmbedConfig(embed_dim=64, hidden_dim=128, epochs=100, lr=0.01, use_edge_weights=True, device=device)

config = PipelineConfig(
    # --- data ---
    dataset_name="cifar10",
    data_root="./data",
    split_config=SplitConfig(train_frac=0.8, test_frac=0.1, embed_frac=0.1, seed=split_seed),
    embed_sample_limit=None,  # None = use the whole embed split

    # --- model ---
    model_name="resnet18",
    small_inputs=True,  # CIFAR-style stem adaptation for small (<=96px) images
    input_size=32,

    # --- baseline training ---
    baseline_train_config=TrainConfig(epochs=20, batch_size=128, lr=0.1, device=device, seed=train_seed),

    # --- activation extraction ---
    activation_config=ActivationConfig(
        stats=["mean", "max", "std", "median", "skew", "kurtosis", "entropy"],
        batch_size=128, device=device,
    ),

    # --- similarity graph ---
    graph_config=GraphConfig(similarity_threshold=0.7, same_layer_only=False, device=device),

    # --- graph embedding ---
    graph_embedding_method=graph_embedding_method,
    node2vec_config=node2vec_config,
    spectral_config=spectral_config,
    raw_embed_config=raw_embed_config,
    diffusion_config=diffusion_config,
    gcn_config=gcn_config,

    # --- clustering ---
    cluster_config=ClusterConfig(method="ward", n_clusters=30, min_cluster_size=3),

    # --- pruning (shared fraction: matched pruning ratio for a fair comparison) ---
    prune_fraction=0.3,
    l2_global_pruning=True,

    # --- recalibration: retrain each pruned model before evaluating it. `epochs` here is
    # the tunable knob CLAUDE.md calls out -- how much recalibration a given prune ratio
    # needs is an open empirical question, so sweep this rather than assume a fixed value.
    recalibration_config=TrainConfig(epochs=5, lr=0.005, device=device, seed=train_seed),

    # --- comparison eval ---
    timing_config=TimingConfig(num_warmup=10, num_trials=30, device=device),
    eval_batch_size=128,
    seed=train_seed,
)

## 3. Run the pipeline

In [ ]:
base = train_base(config)
result = prune_and_compare(base, config)


## 4. Results

`result.baseline` / `result.ours` / `result.l2_baseline` are each a `ModelReport` (`test_metrics`, `flops_params`, `inference`). `result.ours_vs_baseline` / `result.l2_vs_baseline` are compression ratios (>1 means smaller/faster than baseline).

In [ ]:
import pandas as pd

def row(name, report):
    m = report.test_metrics
    return {
        "model": name,
        "accuracy": m["accuracy"],
        "balanced_accuracy": m["balanced_accuracy"],
        "f1": m["f1"],
        "auc": m["auc"],
        "params": report.flops_params.params,
        "ops": report.flops_params.ops,
        "inference_ms": report.inference.mean_time_s * 1000,
        "peak_mem_MB": (report.inference.peak_memory_bytes / (1024**2))
                         if report.inference.peak_memory_bytes else None,
    }

summary = pd.DataFrame([
    row("baseline", result.baseline),
    row("ours", result.ours),
    row("l2_baseline", result.l2_baseline),
])
summary

In [ ]:
print(f"filters: {result.num_filters}  candidates: {result.num_prune_candidates}  "
      f"pruned: {result.num_pruned}  graph edges: {result.graph_num_edges}")
print("ours vs baseline:", result.ours_vs_baseline)
print("l2 vs baseline:  ", result.l2_vs_baseline)

## 5. (Optional) Sweep an ablation

Example: compare all five graph-embedding methods at the same pruning ratio. `train_base` runs once (the baseline model already trained in section 3 is reused as-is); only `prune_and_compare` -- the part each method actually changes -- reruns per method.

In [ ]:
import copy

sweep_results = {}
for method in ["node2vec", "spectral", "raw", "diffusion", "gcn"]:
    sweep_config = copy.deepcopy(config)
    sweep_config.graph_embedding_method = method
    sweep_results[method] = prune_and_compare(base, sweep_config)
    print(method, "->", sweep_results[method].ours_vs_baseline)
